# Actualización y estandarización recurrente de dependencias externas

## Configuración importe de dependencias

In [1]:
%load_ext IPython.extensions.autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path
def find_src_folder(current_path: Path, folder_name: str = 'src') -> Path:
    search_directories = [current_path] + list(current_path.parents)
    for parent in search_directories:
        if parent.name == folder_name:
            return parent.parent
    return current_path
src_path = find_src_folder(Path.cwd(), 'src')
sys.path.append(str(src_path))

# Flujo de preprocesamiento de información

## Importación de librerias necesarias

In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from src.utils import SparkUtils
from pyspark.sql import functions as F, types as T, DataFrame, Window
from pyspark.ml.feature import Tokenizer, StopWordsRemover
import json
import os

In [4]:
spark_utils = SparkUtils('eda')
spark = spark_utils.spark

:: loading settings :: url = jar:file:/mnt/d/Maestr%c3%ada/Amazon%20Reviews%20Code/.venv-linux/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/edgar/.ivy2/cache
The jars for the packages stored in: /home/edgar/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3bbde715-da70-4adc-8446-f49155d4e5f8;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.0.0 in central
	found io.delta#delta-storage;3.0.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 131ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.0.0 from central in [default]
	io.delta#delta-storage;3.0.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   

In [5]:
REGENERATE_TABLES = False

In [6]:
SCHEMA = 'silver.preprocess'
GOLD_SCHEMA = 'gold.premodeling'

## Procesamiento de información de productos

### Cargar información de items

La información de items se encuentra en la zona de consumo bronze al ser una carga de información originalmente en formato .jsonl

In [8]:
meta_items = spark.read.format('delta').load(spark_utils.path('meta_items', 'bronze'))

### Unificar y limpiar información textual preprocesada

In [9]:
from src.utils.preprocessors import CleanWords
clean_words = CleanWords()

#### Procesar información de título

In [10]:
meta_items_title_text_clean = clean_words.transform_default_no_tokenization(
    meta_items,
    input_column='title', output_column = 'title'
).select('parent_asin', 'title')

In [11]:
if REGENERATE_TABLES:
    (
        meta_items_title_text_clean.write
            .format('delta')
            .mode('overwrite')
            .save(
                spark_utils.path(
                    'meta_items_title_text_clean',
                    catalog = SCHEMA
                ),
            )
    )
meta_items_title_text_clean = spark.read.format('delta').load(spark_utils.path(
    'meta_items_title_text_clean',
    catalog = SCHEMA
))

#### Procesar información de descripción

In [12]:
meta_items_description_sentences_text_clean = clean_words.transform_default_no_tokenization_array(
        meta_items,
        column_name='description'
    ).filter(
        F.col('description').isNotNull() &
        (F.size(F.col('description')) > 0)
    ).withColumn(
        'description_paragraph',
        F.explode(F.col('description'))
    ).withColumn(
        'paragraph_number',
        F.row_number().over(
            Window.partitionBy('parent_asin').orderBy('parent_asin')
        ).alias('paragraph_number')
    ).withColumn(
        'description_sentence',
        F.explode(
            F.transform(
                F.split(F.col('description_paragraph'), r"[.!?]"),
                lambda x: F.trim(x)
            )
        )
    ).filter(
        F.col('description_sentence').isNotNull() &
        (F.length(F.col('description_sentence')) > 1)
    ).withColumn(
        'sentence_number',
        F.row_number().over(
            Window.partitionBy('parent_asin', 'paragraph_number').orderBy('parent_asin')
        ).alias('sentence_number')
    ).select(
        'parent_asin', 'description_sentence',
        'paragraph_number', 'sentence_number'
    )

meta_items_description_sentences_text_clean = clean_words.spelling_correction(
    meta_items_description_sentences_text_clean,
    input_column='description_sentence', output_column = 'description_sentence'
)

In [13]:
if REGENERATE_TABLES:
    (
        meta_items_description_sentences_text_clean.write
            .format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .save(
                spark_utils.path(
                    'meta_items_description_sentences_text_clean',
                    catalog = SCHEMA
                ),
            )
    )
meta_items_description_sentences_text_clean = spark.read.format('delta').load(spark_utils.path(
    'meta_items_description_sentences_text_clean',
    catalog = SCHEMA
))

In [14]:
meta_items_descriptions_consolidated = meta_items_description_sentences_text_clean.groupBy('parent_asin').agg(
    F.collect_list('description_sentence').alias('description_sentences')
).withColumn(
    'description_joined',
    F.concat_ws('. ', F.col('description_sentences'))
).select(
    'parent_asin', 'description_joined'
)

In [15]:
if REGENERATE_TABLES:
    (
        meta_items_descriptions_consolidated.write
            .format('delta')
            .mode('overwrite')
            .save(
                spark_utils.path(
                    'meta_items_descriptions_consolidated',
                    catalog = SCHEMA
                ),
            )
    )
meta_items_descriptions_consolidated = spark.read.format('delta').load(spark_utils.path(
    'meta_items_descriptions_consolidated',
    catalog = SCHEMA
))

#### Procesar información de características

In [16]:
meta_items_features_text_clean = clean_words.transform_default_no_tokenization_array(
        meta_items,
        column_name='features'
    ).filter(
        F.col('features').isNotNull() &
        (F.size(F.col('features')) > 0)
    ).withColumn(
        'feature',
        F.explode(F.col('features'))
    ).withColumn(
        'feature_number',
        F.row_number().over(
            Window.partitionBy('parent_asin').orderBy('parent_asin')
        )
    ).withColumn(
        'feature_sentence',
        F.explode(
            F.transform(
                F.split(F.col('feature'), r"[.!?]"),
                lambda x: F.trim(x)
            )
        )
    ).filter(
        F.col('feature_sentence').isNotNull() &
        (F.length(F.col('feature_sentence')) > 1)
    ).withColumn(
        'sentence_number',
        F.row_number().over(
            Window.partitionBy('parent_asin', 'feature_number').orderBy('parent_asin')
        ).alias('sentence_number')
    ).select(
        'parent_asin', 'feature_sentence',
        'feature_number', 'sentence_number'
    )

meta_items_features_text_clean = clean_words.spelling_correction(
    meta_items_features_text_clean,
    input_column='feature_sentence', output_column = 'feature_sentence'
)

In [17]:
if REGENERATE_TABLES:
    (
        meta_items_features_text_clean.write
            .format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .save(
                spark_utils.path(
                    'meta_items_features_text_clean',
                    catalog = SCHEMA
                ),
            )
    )
meta_items_features_text_clean = spark.read.format('delta').load(spark_utils.path(
    'meta_items_features_text_clean',
    catalog = SCHEMA
))

In [18]:
meta_items_features_consolidated = meta_items_features_text_clean.groupBy('parent_asin').agg(
    F.collect_list('feature_sentence').alias('features')
).withColumn(
    'features_joined',
    F.concat_ws('. ', F.col('features'))
).select(
    'parent_asin', 'features_joined'
)

In [19]:
if REGENERATE_TABLES:
    (
        meta_items_features_consolidated.write
            .format('delta')
            .mode('overwrite')
            .save(
                spark_utils.path(
                    'meta_items_features_consolidated',
                    catalog = SCHEMA
                ),
            )
    )
meta_items_features_consolidated = spark.read.format('delta').load(spark_utils.path(
    'meta_items_features_consolidated',
    catalog = SCHEMA
))

#### Unificar información

In [20]:
meta_items_texts_unified = meta_items.alias('A').join(
    meta_items_title_text_clean.alias('B'),
    F.col('A.parent_asin') == F.col('B.parent_asin'),
    'left'
).join(
    meta_items_descriptions_consolidated.alias('C'),
    F.col('A.parent_asin') == F.col('C.parent_asin'),
    'left'
).join(
    meta_items_features_consolidated.alias('D'),
    F.col('A.parent_asin') == F.col('D.parent_asin'),
    'left'
).select(
    F.col('A.parent_asin'),
    F.col('A.main_category'),
    F.col('A.average_rating'),
    F.col('A.rating_number'),
    F.col('A.categories'),
    F.col('A.details'),
    F.col('B.title'),
    F.col('C.description_joined'),
    F.col('D.features_joined'),
    F.format_string(
        "%s %s %s",
        F.when( F.col('B.title').isNull(), F.lit('')).otherwise(F.col('B.title')),
        F.when( F.col('C.description_joined').isNull(), F.lit('')).otherwise(F.col('C.description_joined')),
        F.when( F.col('D.features_joined').isNull(), F.lit('')).otherwise(F.col('D.features_joined')),
    ).alias('colapsed_text')
)

In [21]:
if REGENERATE_TABLES:
    (
        meta_items_texts_unified.write
            .format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .save(
                spark_utils.path(
                    'meta_items_texts_unified',
                    catalog = SCHEMA
                ),
            )
    )
meta_items_texts_unified = spark.read.format('delta').load(spark_utils.path(
    'meta_items_texts_unified',
    catalog = SCHEMA
))

In [22]:
meta_items_texts_unified.show(10)

+-----------+--------------------+--------------+-------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|parent_asin|       main_category|average_rating|rating_number|          categories|             details|               title|  description_joined|     features_joined|       colapsed_text|
+-----------+--------------------+--------------+-------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
| 0007182147|               Books|           4.1|           70|[Electronics, eBo...|{"Publisher":"Fou...|                Solo|                NULL|The highly antici...|Solo  The highly ...|
| 0072826843|               Books|           4.3|           72|[Video Games, PC,...|                  {}|Evolution of the ...|                NULL|Evolution of the ...|Evolution of the ...|
| 0321735722|           Computers|           5.0| 

### Tokenizar texto resultante

In [23]:
from src.utils.preprocessors import CleanWords
clean_words = CleanWords()
meta_items_texts_tokenized = clean_words.tokenize(
    meta_items_texts_unified,
    input_column='colapsed_text', output_column = 'colapsed_text_words'
)

In [24]:
if REGENERATE_TABLES:
    (
        meta_items_texts_tokenized.write
            .format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .save(spark_utils.path(
                'meta_items_texts_tokenized', catalog = SCHEMA
            ))
    )
meta_items_texts_tokenized = spark.read.format('delta').load(spark_utils.path(
    'meta_items_texts_tokenized', catalog = SCHEMA
))

In [25]:
meta_items_texts_tokenized.show(6)

+-----------+-------------+--------------+-------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|parent_asin|main_category|average_rating|rating_number|          categories|             details|               title|  description_joined|     features_joined|       colapsed_text| colapsed_text_words|
+-----------+-------------+--------------+-------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
| 0312363877|        Books|           4.1|           19|[Electronics, eBo...|{"Publisher":"St....|Beyond Hell and B...|From Publishers W...|An inside look at...|Beyond Hell and B...|[beyond, hell, an...|
| 0439215498|     Software|           3.4|            7|[Software, Childr...|{"Is Discontinued...|Clifford The Big ...|                NULL|                NULL|Clifford The Big ...|[c

In [26]:
meta_items_texts_tokenized_with_length = (
    meta_items_texts_tokenized
        .withColumn(
            'colapsed_text_length', F.size(F.col('colapsed_text_words'))
        )
        .select(
            F.col('parent_asin'),
            F.col('main_category'),
            F.col('average_rating'),
            F.col('rating_number'),
            F.col('categories'),
            F.col('details'),
            F.col('description_joined'),
            F.col('features_joined'),
            F.col('title'),
            F.col('colapsed_text'),
            F.col('colapsed_text_words'),
            F.col('colapsed_text_length')
        )
)

In [27]:
if REGENERATE_TABLES:
    (
        meta_items_texts_tokenized_with_length.write
            .format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .save(spark_utils.path(
                'meta_items_texts_tokenized_with_length', catalog = SCHEMA
            ))
    )
meta_items_texts_tokenized_with_length = spark.read.format('delta').load(spark_utils.path(
    'meta_items_texts_tokenized_with_length', catalog = SCHEMA
))

In [28]:
meta_items_texts_tokenized_with_length.show(6)

+-----------+---------------+--------------+-------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|parent_asin|  main_category|average_rating|rating_number|          categories|             details|  description_joined|     features_joined|               title|       colapsed_text| colapsed_text_words|colapsed_text_length|
+-----------+---------------+--------------+-------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
| 0076163997|       Software|           3.0|            1|                  []|{"Package Dimensi...|2009 SRA Imagine ...|                NULL|Imagine It! Photo...|Imagine It! Photo...|[imagine, it!, ph...|                  39|
| 0321636864|       Software|           4.2|            2|[Software, Photog...|{"Package Dim

#### Filtrar textos con 50 tokens o más

In [29]:
meta_items_texts_tokenized_with_length_min_50 = meta_items_texts_tokenized_with_length.filter(
    F.col('colapsed_text_length') >= 50
)

In [30]:
f"Cantidad de productos con 50 tokens o más: {meta_items_texts_tokenized_with_length_min_50.count():,}"

'Cantidad de productos con 50 tokens o más: 2,084,520'

### Filtrar categorías seleccionadas

In [31]:
meta_items_texts_tokenized_categories = meta_items_texts_tokenized_with_length_min_50.filter(
    F.col('main_category').isin(
        'Cell Phones & Accessories', 'Computers', 'All Electronics', 'Camera & Photo', 
        'Home Audio & Theater', 'Industrial & Scientific', 'Car Electronics', 'Amazon Home', 
        'Tools & home improvement', 'Office Products', 'Sports & outdoors'
    ))

In [32]:
f"Cantidad de productos en las categorías seleccionadas: {meta_items_texts_tokenized_categories.count():,}"

'Cantidad de productos en las categorías seleccionadas: 1,670,384'

### Conversión campo detalles

In [33]:
meta_items_texts_tokenized_categories.show(1)

+-----------+--------------------+--------------+-------------+--------------------+--------------------+--------------------+---------------+--------------------+--------------------+--------------------+--------------------+
|parent_asin|       main_category|average_rating|rating_number|          categories|             details|  description_joined|features_joined|               title|       colapsed_text| colapsed_text_words|colapsed_text_length|
+-----------+--------------------+--------------+-------------+--------------------+--------------------+--------------------+---------------+--------------------+--------------------+--------------------+--------------------+
| 011040047X|Cell Phones & Acc...|           1.0|            1|[Cell Phones & Ac...|{"Special feature...|Purple Hard Case ...|           NULL|Purple Hard Case ...|Purple Hard Case ...|[purple, hard, ca...|                 109|
+-----------+--------------------+--------------+-------------+--------------------+--------

In [34]:
from src.utils.dataframe import JsonNormalizer

normalizer = JsonNormalizer()
details_expanded = normalizer.normalize_json_column(
    meta_items_texts_tokenized_categories, 'details', 'parent_asin'
)

In [35]:
if REGENERATE_TABLES:
    (
        details_expanded.write
            .format('delta')
            .mode('overwrite')
            .save(spark_utils.path(
                'details_expanded', catalog = SCHEMA
            ))
    )
details_expanded = spark.read.format('delta').load(spark_utils.path(
    'details_expanded', catalog = SCHEMA
))

In [36]:
details_expanded.show(10)

+-----------+--------------------+--------------------+
|parent_asin|               clave|               valor|
+-----------+--------------------+--------------------+
| 1060264447|  Product Dimensions|1 x 0.1 x 0.5 inches|
| 1060264447|         Item Weight|          1.5 Ounces|
| 1060264447|        Manufacturer|      Factory Direct|
| 1060264447|   Item model number|                64GB|
| 1060264447|Date First Available|       June 30, 2014|
| 1060264447|               Brand|      Factory Direct|
| 1060264447|   Flash Memory Type|          Micro SDHC|
| 1060264447|Memory Storage Ca...|               64 GB|
| 1060264447|Secure Digital As...|            Class 10|
| 1060698978|  Product Dimensions|   24 x 6 x 6 inches|
+-----------+--------------------+--------------------+
only showing top 10 rows



## Procesamiento de información de reseñas

### Unificación información textual de reseñas

In [37]:
meta_items_texts_tokenized_categories.count()

1670384

In [38]:
reviews = spark.read.format('delta').load(spark_utils.path('reviews', 'bronze'))

In [39]:
associated_reviews = (
    reviews.alias('A')
        .join(
            meta_items_texts_tokenized_categories.alias('B'),
            F.col('A.parent_asin') == F.col('B.parent_asin')
        )
        .select(
            F.col('A.*'),
        )
)

In [40]:
associated_reviews.count()

51465009

In [41]:
from src.utils.preprocessors import CleanWords
clean_words = CleanWords()
reviews_title_clean = clean_words.transform_default_no_tokenization(
    associated_reviews,
    input_column='title', output_column = 'title'
)

In [42]:
from src.utils.preprocessors import CleanWords
clean_words = CleanWords()
reviews_description_clean = clean_words.transform_default_no_tokenization(
    reviews_title_clean,
    input_column='text', output_column = 'text'
)

In [43]:
reviews_text_unified = (
    reviews_description_clean
        .select(
            F.col('*'),
            F.concat_ws('. ', F.col('title'), F.col('text')).alias('text_unified')
        )
)

In [44]:
reviews_text_unified.count()

51465009

### Tokenizar texto resultante

In [45]:
from src.utils.preprocessors import CleanWords
clean_words = CleanWords()
reviews_text_spelled = clean_words.spelling_correction(
    reviews_text_unified,
    input_column='text_unified', output_column = 'text_unified_spelled'
)
reviews_text_normalized = clean_words.normalize_text(
    reviews_text_spelled,
    column_name='text_unified_spelled'
)
reviews_text_tokenized = clean_words.tokenize(
    reviews_text_normalized,
    input_column='text_unified_spelled', output_column = 'text_unified_words'
)

In [46]:
if REGENERATE_TABLES:
    (
        reviews_text_tokenized.write
            .format('delta')
            .mode('overwrite')
            .save(spark_utils.path(
                'reviews_text_tokenized', catalog = SCHEMA
            ))
    )
reviews_text_tokenized = spark.read.format('delta').load(spark_utils.path(
    'reviews_text_tokenized', catalog = SCHEMA
))

In [47]:
reviews_text_tokenized_with_length = (
    reviews_text_tokenized
        .withColumn(
            'text_unified_length', F.size(F.col('text_unified_words'))
        )
)

In [48]:
if REGENERATE_TABLES:
    (
        reviews_text_tokenized_with_length.write
            .format('delta')
            .mode('overwrite')
            .save(spark_utils.path(
                'reviews_text_tokenized_with_length', catalog = SCHEMA
            ))
    )
reviews_text_tokenized_with_length = spark.read.format('delta').load(spark_utils.path(
    'reviews_text_tokenized_with_length', catalog = SCHEMA
))

### Filtrar reseñas con más de 30 tokens en total

In [49]:
reviews_text_tokenized_with_length_min_30 = reviews_text_tokenized_with_length.filter(
    F.col('text_unified_length') >= 30
)

In [50]:
reviews_text_tokenized_with_length_min_30.show(1)

+------+-----------+--------------------+-------------+------------+-----------+------+--------------------+--------------------+--------------------+-------------------+
|rating|      title|                text|    timestamp|helpful_vote|parent_asin|images|        text_unified|text_unified_spelled|  text_unified_words|text_unified_length|
+------+-----------+--------------------+-------------+------------+-----------+------+--------------------+--------------------+--------------------+-------------------+
|   1.0|Frauds!!!!!|Would give them 0...|1443316881000|           2| 1059844729|    []|Frauds!!!!!. Woul...|frauds       woul...|[frauds, would, g...|                126|
+------+-----------+--------------------+-------------+------------+-----------+------+--------------------+--------------------+--------------------+-------------------+
only showing top 1 row



### Transformación puntuación a formato booleano

In [51]:
reviews_fixed_rating = (
    reviews_text_tokenized_with_length_min_30
        .withColumn(
            'rating_boolean',
            F.when(F.col('rating').isin(0, 1, 2, 3), F.lit(0))
                .otherwise(F.lit(1))
        )
)

In [52]:
reviews_fixed_rating.count()

24091496

In [53]:
if REGENERATE_TABLES:
    (
        reviews_fixed_rating.write
            .format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .save(spark_utils.path(
                'reviews_fixed_rating', catalog = SCHEMA
            ))
    )
reviews_fixed_rating = spark.read.format('delta').load(spark_utils.path(
    'reviews_fixed_rating', catalog = SCHEMA
))

In [54]:
reviews_fixed_rating.show(1)

+------+------------------+--------------------+-------------+------------+-----------+------+--------------------+--------------------+--------------------+-------------------+--------------+
|rating|             title|                text|    timestamp|helpful_vote|parent_asin|images|        text_unified|text_unified_spelled|  text_unified_words|text_unified_length|rating_boolean|
+------+------------------+--------------------+-------------+------------+-----------+------+--------------------+--------------------+--------------------+-------------------+--------------+
|   5.0|Thank you teacher!|Thank you teacher...|1391475835000|           0| 098536081X|    []|Thank you teacher...|thank you teacher...|[thank, you, teac...|                 30|             1|
+------+------------------+--------------------+-------------+------------+-----------+------+--------------------+--------------------+--------------------+-------------------+--------------+
only showing top 1 row



## Procesamiento conjunto de reseñas y productos

### FIltrar productos con al menos 5 reseñas

In [55]:
products_with_at_least_5_reviews = (
    reviews_fixed_rating
        .groupBy('parent_asin')
        .agg(F.count('*').alias('review_count'))
        .filter(F.col('review_count') >= 5)
)

meta_items_texts_tokenized_categories_with_at_least_5_reviews = (
    meta_items_texts_tokenized_categories.alias('A')
        .join(
            products_with_at_least_5_reviews.alias('B'),
            F.col('A.parent_asin') == F.col('B.parent_asin'),
            'inner'
        )
        .select(
            F.col('A.*'),
            F.col('B.review_count')
        )
)


In [56]:
if REGENERATE_TABLES:
    (
        meta_items_texts_tokenized_categories_with_at_least_5_reviews.write
            .format('delta')
            .mode('overwrite')
            .save(spark_utils.path(
                'meta_items_texts_tokenized_categories_with_at_least_5_reviews', catalog = SCHEMA
            ))
    )
meta_items_texts_tokenized_categories_with_at_least_5_reviews = spark.read.format('delta').load(spark_utils.path(
    'meta_items_texts_tokenized_categories_with_at_least_5_reviews', catalog = SCHEMA
))

In [57]:
f"Cantidad de productos con al menos 5 reseñas: {meta_items_texts_tokenized_categories_with_at_least_5_reviews.count():,}"

'Cantidad de productos con al menos 5 reseñas: 456,530'

### Codificación de campos categóricos (subcategorías y lista de categorías)

In [58]:
from src.utils.dataframe import OneHotColumnEncoder

encoder = OneHotColumnEncoder()
main_category_encoded = encoder.one_hot_encode(
    meta_items_texts_tokenized_categories_with_at_least_5_reviews, 
    input_column='main_category', output_prefix='main_category'
)

In [59]:
if REGENERATE_TABLES:
    (
        main_category_encoded.write
            .format('delta')
            .mode('overwrite')
            .save(spark_utils.path(
                'main_category_encoded', catalog = SCHEMA
            ))
    )
main_category_encoded = spark.read.format('delta').load(spark_utils.path(
    'main_category_encoded', catalog = SCHEMA
))

In [60]:
main_category_encoded.show(10)

+--------------------+--------------------+--------+-----------+--------------+-------------+-----+-----+-----------+--------------------+--------------------+------+--------------------+--------------------+--------------------+---------------------+--------------------+--------------------+------------+-----------------------+-----------------------------+--------------------------------+-------------------------+-----------------------------------+-------------------------------------+-----------------------------+-----------------------------+--------------------------+
|               title|       main_category|features|description|average_rating|rating_number|price|store|parent_asin|          categories|             details|images|description_colapsed|   features_colapsed|       colapsed_text|colapsed_text_spelled| colapsed_text_words|colapsed_text_length|review_count|main_category_computers|main_category_all_electronics|main_category_home_audio_theater|main_category_amazon_home|

In [61]:
main_category_encoded_categories = main_category_encoded.withColumn(
    "category", F.explode('categories')
)

In [62]:
if REGENERATE_TABLES:
    (
        main_category_encoded_categories.write
            .format('delta')
            .mode('overwrite')
            .save(spark_utils.path(
                'main_category_encoded_categories', catalog = SCHEMA
            ))
    )
main_category_encoded_categories = spark.read.format('delta').load(spark_utils.path(
    'main_category_encoded_categories', catalog = SCHEMA
))

In [63]:
main_category_encoded_categories.count()

1790345

### Filtrado de reseñas ùnicas para productos seleccionados

In [64]:
reviews_chosen = (
    meta_items_texts_tokenized_categories_with_at_least_5_reviews.alias('A')
        .join(
            reviews_fixed_rating.alias('B'),
            F.col('A.parent_asin') == F.col('B.parent_asin'),
            'inner'
        )
        .select('B.*')
)

In [65]:
if REGENERATE_TABLES:
    (
        reviews_chosen.write
            .format('delta')
            .mode('overwrite')
            .save(spark_utils.path(
                'reviews_chosen', catalog = SCHEMA
            ))
    )
reviews_chosen = spark.read.format('delta').load(spark_utils.path(
    'reviews_chosen', catalog = SCHEMA
))
reviews_chosen.count()

22718212

In [66]:
reviews_chosen.select('parent_asin').distinct().count()

456530

## Selección de conjuntos textuales y tablas finales

In [67]:
reviews_chosen.count()

22718212

In [68]:
if REGENERATE_TABLES:
    (
        main_category_encoded.write
            .format('delta')
            .mode('overwrite')
            .save(spark_utils.path(
                'main_category_encoded', catalog = GOLD_SCHEMA
            ))
    )
main_category_encoded = spark.read.format('delta').load(spark_utils.path(
    'main_category_encoded', catalog = GOLD_SCHEMA
))

In [69]:
if REGENERATE_TABLES:
    (
        reviews_chosen.withColumn(
            'review_id', 
            F.row_number().over(Window.orderBy(F.col('parent_asin').asc()))
        )
            .write
            .format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .save(spark_utils.path(
                'reviews_chosen', catalog = GOLD_SCHEMA
            ))
    )
reviews_chosen = spark.read.format('delta').load(spark_utils.path(
    'reviews_chosen', catalog = GOLD_SCHEMA
))

In [70]:
if REGENERATE_TABLES:
    (
        meta_items_title_text_clean.alias('A')
            .join(
                meta_items_texts_tokenized_categories_with_at_least_5_reviews.alias('B'),
                F.col('A.parent_asin') == F.col('B.parent_asin'),
                'inner'
            ).select('A.*', F.row_number().over(Window.orderBy(F.col('A.parent_asin').asc())).alias('record_id'))
            .write
            .format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .save(spark_utils.path(
                'meta_items_title_text_clean', catalog = GOLD_SCHEMA
            ))
    )
meta_items_title_text_clean = spark.read.format('delta').load(spark_utils.path(
    'meta_items_title_text_clean', catalog = GOLD_SCHEMA
))

In [71]:
if REGENERATE_TABLES:
    (
        meta_items_description_sentences_text_clean.alias('A')
            .join(
                meta_items_texts_tokenized_categories_with_at_least_5_reviews.alias('B'),
                F.col('A.parent_asin') == F.col('B.parent_asin'),
                'inner'
            ).select('A.*', F.row_number().over(Window.orderBy(F.col('A.parent_asin').asc())).alias('record_id'))
            .write
            .format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .save(spark_utils.path(
                'meta_items_description_sentences_text_clean', catalog = GOLD_SCHEMA
            ))
    )
meta_items_description_sentences_text_clean = spark.read.format('delta').load(spark_utils.path(
    'meta_items_description_sentences_text_clean', catalog = GOLD_SCHEMA
))

In [72]:
if REGENERATE_TABLES:
    (
        meta_items_features_text_clean.alias('A').join(
                meta_items_texts_tokenized_categories_with_at_least_5_reviews.alias('B'),
                F.col('A.parent_asin') == F.col('B.parent_asin'),
                'inner'
            ).select('A.*', F.row_number().over(Window.orderBy(F.col('A.parent_asin').asc())).alias('record_id'))
            .write
            .format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .save(spark_utils.path(
                'meta_items_features_text_clean', catalog = GOLD_SCHEMA
            ))
    )
meta_items_features_text_clean = spark.read.format('delta').load(spark_utils.path(
    'meta_items_features_text_clean', catalog = GOLD_SCHEMA
))

In [73]:
if REGENERATE_TABLES:
    (
        details_expanded.alias('A').join(
                meta_items_texts_tokenized_categories_with_at_least_5_reviews.alias('B'),
                F.col('A.parent_asin') == F.col('B.parent_asin'),
                'inner'
            ).select('A.*', F.row_number().over(Window.orderBy(F.col('A.parent_asin').asc())).alias('record_id'))
            .write
            .format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .save(spark_utils.path(
                'details_expanded', catalog = GOLD_SCHEMA
            ))
    )
details_expanded = spark.read.format('delta').load(spark_utils.path(
    'details_expanded', catalog = GOLD_SCHEMA
))

## Muestreo de productos en función de categorías

In [74]:
from src.utils.dataframe import SampleDataset
sample_dataset = SampleDataset()
df_items_sample_balanced = sample_dataset.sample_dataset_weighted(
    main_category_encoded, 'main_category', 
    scale = 2
)

Counts: [Row(main_category='computers', count=92874), Row(main_category='cell_phones_accessories', count=157722), Row(main_category='office_products', count=4583), Row(main_category='camera_photo', count=47079), Row(main_category='industrial_scientific', count=10148), Row(main_category='car_electronics', count=5454), Row(main_category='amazon_home', count=4163), Row(main_category='home_audio_theater', count=25586), Row(main_category='all_electronics', count=108921)]
Sampling ratios: {'computers': 1.698236320175722, 'cell_phones_accessories': 1.0, 'office_products': 34.414575605498584, 'camera_photo': 3.350156120563309, 'industrial_scientific': 15.542175798186836, 'car_electronics': 28.91859185918592, 'amazon_home': 37.88662022579871, 'home_audio_theater': 6.164386774017041, 'all_electronics': 1.4480403228027654}
Counts balanced: [Row(main_category='computers', count=8395), Row(main_category='cell_phones_accessories', count=8455), Row(main_category='office_products', count=4583), Row(ma

In [75]:
if REGENERATE_TABLES:
    (
        df_items_sample_balanced.write
            .format('delta')
            .mode('overwrite')
            .save(spark_utils.path(
                'df_items_sample_balanced', catalog = GOLD_SCHEMA))
    )
df_items_sample_balanced = spark.read.format('delta').load(spark_utils.path(
    'df_items_sample_balanced', catalog = GOLD_SCHEMA
))

## Generación oraciones de reseñas

### Indexación de conjunto de reseñas

In [76]:
reviews_chosen.head(10)

[Row(rating=1.0, title='TOPO no longer supports Mac!', text="After experiencing some problems with the route planning function, I contacted National Geo TOPO support.  I was very bluntly informed that TOPO no longer supported recent Mac OS.  Tough luck you're on your own!", timestamp='1355348004000', helpful_vote=2, parent_asin='1597750328', images=[], text_unified="TOPO no longer supports Mac!. After experiencing some problems with the route planning function, I contacted National Geo TOPO support.  I was very bluntly informed that TOPO no longer supported recent Mac OS.  Tough luck you're on your own!", text_unified_spelled='topo no longer supports mac   after experiencing some problems with the route planning function  i contacted national geo topo support   i was very bluntly informed that topo no longer supported recent mac os   tough luck you re on your own ', text_unified_words=['topo', 'no', 'longer', 'supports', 'mac', 'after', 'experiencing', 'some', 'problems', 'with', 'the'

In [77]:

reviews_indexed_tmp = (
    reviews_chosen
        .select(
            'title',
            'text',
            'parent_asin',
            'rating',
            'helpful_vote',
            'rating_boolean',
            F.monotonically_increasing_id().alias('review_id')
        )
)

In [78]:
if REGENERATE_TABLES:
    (
        reviews_indexed_tmp.write
            .format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .save(spark_utils.path(
                'reviews_indexed_tmp', catalog = SCHEMA
            ))
    )
reviews_indexed_tmp = spark.read.format('delta').load(spark_utils.path(
    'reviews_indexed_tmp', catalog = SCHEMA
))

In [79]:
reviews_indexed_successive = reviews_indexed_tmp.select(
        'review_id', F.row_number().over(Window.orderBy(F.lit(0))).alias('review_id_index')
    )

if REGENERATE_TABLES:
    (
        reviews_indexed_successive.write
            .format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .save(spark_utils.path(
                'reviews_indexed_successive', catalog = SCHEMA
            ))
    )
reviews_indexed_successive = spark.read.format('delta').load(spark_utils.path(
    'reviews_indexed_successive', catalog = SCHEMA
))

In [80]:
reviews_indexed_successive.head(10)

[Row(review_id=0, review_id_index=1),
 Row(review_id=1563368144698, review_id_index=2),
 Row(review_id=1, review_id_index=3),
 Row(review_id=1563368144699, review_id_index=4),
 Row(review_id=2, review_id_index=5),
 Row(review_id=1563368144700, review_id_index=6),
 Row(review_id=3, review_id_index=7),
 Row(review_id=1563368144701, review_id_index=8),
 Row(review_id=4, review_id_index=9),
 Row(review_id=1563368144702, review_id_index=10)]

In [81]:
reviews_indexed = (
    reviews_indexed_successive.alias('A')
        .join(
            reviews_indexed_tmp.alias('B'),
            F.col('A.review_id') == F.col('B.review_id'),
            'left'
        ).select(
            'title',
            'text',
            'parent_asin',
            'rating',
            'helpful_vote',
            'rating_boolean',
            F.col('A.review_id_index').alias('review_id')
        )
)

In [82]:
if REGENERATE_TABLES:
    (
        reviews_indexed.write
            .format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .save(spark_utils.path(
                'reviews_indexed', catalog = SCHEMA
            ))
    )
reviews_indexed = spark.read.format('delta').load(spark_utils.path(
    'reviews_indexed', catalog = SCHEMA
))

In [83]:
reviews_indexed_sentences = reviews_indexed.withColumn(
        'text_sentence',
        F.explode(
            F.transform(
                F.split(F.col('text'), r"[.!?]"),
                lambda x: F.trim(x)
            )
        )
    ).filter(
        F.col('text_sentence').isNotNull() &
        (F.length(F.col('text_sentence')) > 1)
    ).withColumn(
        'sentence_number',
        F.row_number().over(
            Window.partitionBy('review_id').orderBy('review_id')
        ).alias('sentence_number')
    ).withColumn(
        'record_id',
        F.monotonically_increasing_id()
    ).select(
        'review_id', 'text_sentence',
        'sentence_number', 'record_id'
    )

reviews_indexed_sentences = clean_words.spelling_correction(
    reviews_indexed_sentences,
    input_column='text_sentence', output_column = 'text_sentence'
).orderBy('record_id')

In [84]:
reviews_indexed_sentences_items_sample_balanced = reviews_indexed_sentences.alias('A').join(
    reviews_indexed.alias('B'),
    on = 'review_id',
    how = 'inner'
).join(
    df_items_sample_balanced.alias('C'),
    on = 'parent_asin',
    how = 'inner'
).select('A.*')

In [ ]:
if REGENERATE_TABLES:
    (
        reviews_indexed_sentences_items_sample_balanced.write
            .format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .save(spark_utils.path(
                'reviews_indexed_sentences_items_sample_balanced', catalog = GOLD_SCHEMA
            ))
    )
reviews_indexed_sentences_items_sample_balanced = spark.read.format('delta').load(spark_utils.path(
    'reviews_indexed_sentences_items_sample_balanced', catalog = GOLD_SCHEMA
))

In [1]:
REGENERATE_REVIEWS_SENTENCES = True
BATCH_SIZE = 1_000_000

In [ ]:
if REGENERATE_REVIEWS_SENTENCES:
    try:
        existing_data = spark.read.format('delta').load(spark_utils.path(
            'reviews_indexed_sentences_items_sample_balanced', catalog = GOLD_SCHEMA
        ))
        max_review_id_saved = existing_data.select('review_id').agg(F.max('review_id')).collect()[0][0]
        if max_review_id_saved is None:
            max_review_id_saved = 0
    except:
        max_review_id_saved = 0
    
    min_review_id = reviews_indexed.select('review_id').agg(F.min('review_id')).collect()[0][0]
    max_review_id = reviews_indexed.select('review_id').agg(F.max('review_id')).collect()[0][0]
    
    start_id = max(min_review_id, max_review_id_saved)
    
    for batch_start in range(start_id, max_review_id + 1, BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE - 1, max_review_id)
        
        batch_data = reviews_indexed_sentences.filter(
            (F.col('review_id') >= batch_start) & (F.col('review_id') <= batch_end)
        )
        
        if max_review_id_saved == 0 and batch_start == start_id:
            batch_data.write \
                .format('delta') \
                .mode('overwrite') \
                .option('overwriteSchema', 'true') \
                .save(spark_utils.path(
                    'reviews_indexed_sentences', catalog = GOLD_SCHEMA
                ))
        else:
            batch_data.write \
                .format('delta') \
                .mode('append') \
                .save(spark_utils.path(
                    'reviews_indexed_sentences', catalog = GOLD_SCHEMA
                ))

        current_rows = spark.read.format('delta').load(spark_utils.path(
            'reviews_indexed_sentences', catalog = GOLD_SCHEMA
        ))
        print(f"Batch {batch_start}-{batch_end} saved. Current rows: {current_rows.count():,}")

reviews_indexed_sentences = spark.read.format('delta').load(spark_utils.path(
    'reviews_indexed_sentences', catalog = GOLD_SCHEMA
))

In [86]:
if REGENERATE_REVIEWS_SENTENCES:
    try:
        existing_data = spark.read.format('delta').load(spark_utils.path(
            'reviews_indexed_sentences', catalog = GOLD_SCHEMA
        ))
        max_review_id_saved = existing_data.select('review_id').agg(F.max('review_id')).collect()[0][0]
        if max_review_id_saved is None:
            max_review_id_saved = 0
    except:
        max_review_id_saved = 0
    
    min_review_id = reviews_indexed.select('review_id').agg(F.min('review_id')).collect()[0][0]
    max_review_id = reviews_indexed.select('review_id').agg(F.max('review_id')).collect()[0][0]
    
    start_id = max(min_review_id, max_review_id_saved)
    
    for batch_start in range(start_id, max_review_id + 1, BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE - 1, max_review_id)
        
        batch_data = reviews_indexed_sentences.filter(
            (F.col('review_id') >= batch_start) & (F.col('review_id') <= batch_end)
        )
        
        if max_review_id_saved == 0 and batch_start == start_id:
            batch_data.write \
                .format('delta') \
                .mode('overwrite') \
                .option('overwriteSchema', 'true') \
                .save(spark_utils.path(
                    'reviews_indexed_sentences', catalog = GOLD_SCHEMA
                ))
        else:
            batch_data.write \
                .format('delta') \
                .mode('append') \
                .save(spark_utils.path(
                    'reviews_indexed_sentences', catalog = GOLD_SCHEMA
                ))

        current_rows = spark.read.format('delta').load(spark_utils.path(
            'reviews_indexed_sentences', catalog = GOLD_SCHEMA
        ))
        print(f"Batch {batch_start}-{batch_end} saved. Current rows: {current_rows.count():,}")

reviews_indexed_sentences = spark.read.format('delta').load(spark_utils.path(
    'reviews_indexed_sentences', catalog = GOLD_SCHEMA
))

Batch 2000000-2999999 saved. Current rows: 16,617,867


Batch 3000000-3999999 saved. Current rows: 22,058,940


Batch 4000000-4999999 saved. Current rows: 27,576,374


Batch 5000000-5999999 saved. Current rows: 33,161,983


23:18:26.641 [Thread-4] ERROR org.apache.spark.sql.delta.util.NonFateSharingFuture - Failed to get result from future
scala.runtime.NonLocalReturnControl: null
Batch 6000000-6999999 saved. Current rows: 38,658,953


...

Batch 18000000-18999999 saved. Current rows: 105,267,639


Batch 19000000-19999999 saved. Current rows: 110,786,469


Batch 20000000-20999999 saved. Current rows: 116,304,813


Batch 21000000-21999999 saved. Current rows: 121,850,840


Batch 22000000-22718212 saved. Current rows: 125,844,492


In [87]:
reviews_indexed_sentences.limit(10).toPandas()

,review_id,text_sentence,sentence_number,record_id
0,15852836,After wrestling the television free of the mou...,4,274878342586
1,15853134,A very nice head unit,1,274878342587
2,15853134,Set up with a 5 channel Rockford Fosgate amp,2,274878342588
3,15853134,Sound is amazing,3,274878342589
4,15853134,I have all JBL Power Series speakers can't go ...,4,274878342590
5,15853379,This is the second TCL Roku TV I have purchased,1,274878342591
6,15853379,"I've had my first one for two years, use it a ...",2,274878342592
7,15853379,"plus, the price is great",3,274878342593
8,15853379,I love how I can put in all my streaming accou...,4,274878342594
9,15853379,I don't bother with cable any more,5,274878342595


## Muestreo de productos en función de categorías

In [88]:
from src.utils.dataframe import SampleDataset
sample_dataset = SampleDataset()
df_items_sample_balanced = sample_dataset.sample_dataset_weighted(
    main_category_encoded, 'main_category', 
    scale = 2
)

Counts: [Row(main_category='computers', count=92874), Row(main_category='cell_phones_accessories', count=157722), Row(main_category='office_products', count=4583), Row(main_category='camera_photo', count=47079), Row(main_category='industrial_scientific', count=10148), Row(main_category='car_electronics', count=5454), Row(main_category='amazon_home', count=4163), Row(main_category='home_audio_theater', count=25586), Row(main_category='all_electronics', count=108921)]
Sampling ratios: {'computers': 1.698236320175722, 'cell_phones_accessories': 1.0, 'office_products': 34.414575605498584, 'camera_photo': 3.350156120563309, 'industrial_scientific': 15.542175798186836, 'car_electronics': 28.91859185918592, 'amazon_home': 37.88662022579871, 'home_audio_theater': 6.164386774017041, 'all_electronics': 1.4480403228027654}
Counts balanced: [Row(main_category='computers', count=8395), Row(main_category='cell_phones_accessories', count=8455), Row(main_category='office_products', count=4583), Row(ma

In [7]:
if REGENERATE_TABLES:
    (
        df_items_sample_balanced.write
            .format('delta')
            .mode('overwrite')
            .save(spark_utils.path(
                'df_items_sample_balanced', catalog = GOLD_SCHEMA))
    )
df_items_sample_balanced = spark.read.format('delta').load(spark_utils.path(
    'df_items_sample_balanced', catalog = GOLD_SCHEMA
))

In [8]:
df_items_sample_balanced.count()

64618